# Synthetic Noisy CIFAR-10/100 Dataset Generator

This notebook generates synthetically noisy CIFAR-10/100 datasets following the anomaly types described in the project README:

- **Clean**: A clean image with a correct label
- **Near-miss label noise**: Flip label to semantically similar class (e.g., dog→cat, truck→car)
- **Gross label noise**: Flip label to semantically distant class (e.g., frog→airplane)
- **Out-of-Distribution (OOD)**: Replace image with OOD image and assign random label
- **Clean but Hard**: A clean image that is difficult to classify
- **Random Flip**: Randomly flip label to any other class with equal probability

The notebook also cross-references samples with CIFAR-10/100-N (human annotations) to add a [human/non-human] label column.

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import random

# Paths
CIFAR10_DATA_PATH = Path("/projectnb/ivc-ml/appledora/soft-sharing/data/cifar-10-batches-py")
CIFAR100_DATA_PATH = Path("/projectnb/ivc-ml/appledora/soft-sharing/data/cifar-100-python")
CIFAR10N_PATH = Path("/projectnb/ivc-ml/appledora/CS506/cifar-10-100n/data/CIFAR-10_human_ordered.npy")
CIFAR100N_PATH = Path("/projectnb/ivc-ml/appledora/CS506/cifar-10-100n/data/CIFAR-100_human_ordered.npy")
OUTPUT_DIR = Path("/projectnb/ivc-ml/appledora/CS506/cs506-anomaly-det/data")

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Fixed random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

## CIFAR-10/100 Class Names and Semantic Similarity Maps

In [3]:
# CIFAR-10 class names
CIFAR10_CLASSES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

# CIFAR-100 class names (fine labels)
CIFAR100_CLASSES = [
    'apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle', 
    'bicycle', 'bottle', 'bowl', 'boy', 'bridge', 'bus', 'butterfly', 'camel', 
    'can', 'castle', 'caterpillar', 'cattle', 'chair', 'chimpanzee', 'clock', 
    'cloud', 'cockroach', 'couch', 'crab', 'crocodile', 'cup', 'dinosaur', 
    'dolphin', 'elephant', 'flatfish', 'forest', 'fox', 'girl', 'hamster', 
    'house', 'kangaroo', 'keyboard', 'lamp', 'lawn_mower', 'leopard', 'lion',
    'lizard', 'lobster', 'man', 'maple_tree', 'motorcycle', 'mountain', 'mouse',
    'mushroom', 'oak_tree', 'orange', 'orchid', 'otter', 'palm_tree', 'pear',
    'pickup_truck', 'pine_tree', 'plain', 'plate', 'poppy', 'porcupine', 
    'possum', 'rabbit', 'raccoon', 'ray', 'road', 'rocket', 'rose', 'sea', 
    'seal', 'shark', 'shrew', 'skunk', 'skyscraper', 'snail', 'snake', 'spider', 
    'squirrel', 'streetcar', 'sunflower', 'sweet_pepper', 'table', 'tank', 
    'telephone', 'television', 'tiger', 'tractor', 'train', 'trout', 'tulip', 
    'turtle', 'wardrobe', 'whale', 'willow_tree', 'wolf', 'woman', 'worm'
]

# CIFAR-10 semantic similarity pairs (for near-miss noise)
# Based on semantic similarity and visual similarity
CIFAR10_NEAR_MISS_PAIRS = {
    0: [1],      # airplane -> automobile (vehicles)
    1: [0],      # automobile -> airplane (vehicles)
    2: [4],      # bird -> deer (animals, but more like bird->other animals)
    3: [5],      # cat -> dog (similar animals)
    4: [2, 5],   # deer -> bird, dog (animals)
    5: [3, 4],   # dog -> cat, deer (similar animals)
    6: [4, 5],   # frog -> deer, dog (animals)
    7: [4, 5],   # horse -> deer, dog (similar animals)
    8: [9],      # ship -> truck (vehicles)
    9: [1, 8],   # truck -> automobile, ship (vehicles)
}

# CIFAR-10 gross noise pairs (semantically distant)
CIFAR10_GROSS_NOISE_PAIRS = {
    0: [6, 7],   # airplane -> frog, horse
    1: [2, 6],   # automobile -> bird, frog
    2: [8, 9],   # bird -> ship, truck
    3: [0, 8],   # cat -> airplane, ship
    4: [0, 9],   # deer -> airplane, truck
    5: [0, 8],   # dog -> airplane, ship
    6: [0, 1],   # frog -> airplane, automobile
    7: [2, 8],   # horse -> bird, ship
    8: [2, 3],   # ship -> bird, cat
    9: [2, 6],   # truck -> bird, frog
}

# CIFAR-100 coarse labels (for semantic similarity)
CIFAR100_COARSE_LABELS = {
    'aquatic_mammals': ['beaver', 'dolphin', 'otter', 'seal', 'whale'],
    'fish': ['aquarium_fish', 'flatfish', 'ray', 'shark', 'trout'],
    'flowers': ['orchid', 'poppy', 'rose', 'sunflower', 'tulip'],
    'food_containers': ['bottle', 'bowl', 'can', 'cup', 'plate'],
    'fruit_and_vegetables': ['apple', 'mushroom', 'orange', 'pear', 'sweet_pepper'],
    'household_electrical_devices': ['clock', 'keyboard', 'lamp', 'telephone', 'television'],
    'household_furniture': ['bed', 'chair', 'couch', 'table', 'wardrobe'],
    'insects': ['bee', 'beetle', 'butterfly', 'cockroach', 'caterpillar'],
    'large_carnivores': ['bear', 'leopard', 'lion', 'tiger', 'wolf'],
    'large_man-made_outdoor_things': ['bridge', 'castle', 'house', 'road', 'skyscraper'],
    'large_natural_outdoor_scenes': ['cloud', 'forest', 'mountain', 'plain', 'sea'],
    'large_omnivores_and_herbivores': ['camel', 'cattle', 'chimpanzee', 'elephant', 'kangaroo'],
    'medium_mammals': ['fox', 'porcupine', 'possum', 'raccoon', 'skunk'],
    'non-insect_invertebrates': ['crab', 'lobster', 'snail', 'spider', 'worm'],
    'people': ['baby', 'boy', 'girl', 'man', 'woman'],
    'reptiles': ['crocodile', 'dinosaur', 'lizard', 'snake', 'turtle'],
    'small_mammals': ['hamster', 'mouse', 'rabbit', 'shrew', 'squirrel'],
    'trees': ['maple_tree', 'oak_tree', 'palm_tree', 'pine_tree', 'willow_tree'],
    'vehicles_1': ['bicycle', 'bus', 'lawn_mower', 'motorcycle', 'pickup_truck'],
    'vehicles_2': ['train', 'tractor', 'tank', 'rocket', 'streetcar'],
}

# Create reverse mapping: class name -> coarse label
CIFAR100_CLASS_TO_COARSE = {}
for coarse, fines in CIFAR100_COARSE_LABELS.items():
    for fine in fines:
        CIFAR100_CLASS_TO_COARSE[fine] = coarse

# Create class name to index mapping
CIFAR100_CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(CIFAR100_CLASSES)}

# Create CIFAR-100 near-miss pairs based on coarse labels
CIFAR100_NEAR_MISS_PAIRS = {}
for idx, class_name in enumerate(CIFAR100_CLASSES):
    coarse = CIFAR100_CLASS_TO_COARSE[class_name]
    similar_classes = [CIFAR100_CLASS_TO_IDX[c] for c in CIFAR100_COARSE_LABELS[coarse] if c != class_name]
    CIFAR100_NEAR_MISS_PAIRS[idx] = similar_classes

print(f"CIFAR-10 classes: {len(CIFAR10_CLASSES)}")
print(f"CIFAR-100 classes: {len(CIFAR100_CLASSES)}")
print(f"CIFAR-100 coarse categories: {len(CIFAR100_COARSE_LABELS)}")

CIFAR-10 classes: 10
CIFAR-100 classes: 100
CIFAR-100 coarse categories: 20


## Load CIFAR-10/100 Data

In [4]:
def load_cifar10(data_path: Path) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """Load CIFAR-10 training data from all 5 batches."""
    images = []
    labels = []
    filenames = []
    
    for i in range(1, 6):
        batch_file = data_path / f"data_batch_{i}"
        with open(batch_file, 'rb') as f:
            batch = pickle.load(f, encoding='bytes')
            images.append(batch[b'data'])
            labels.extend(batch[b'labels'])
            filenames.extend([fn.decode('utf-8') for fn in batch[b'filenames']])
    
    images = np.vstack(images)
    labels = np.array(labels)
    
    return images, labels, filenames


def load_cifar100(data_path: Path) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """Load CIFAR-100 training data."""
    with open(data_path / "train", 'rb') as f:
        data = pickle.load(f, encoding='bytes')
        images = data[b'data']
        fine_labels = np.array(data[b'fine_labels'])
        coarse_labels = np.array(data[b'coarse_labels'])
        filenames = [fn.decode('utf-8') for fn in data[b'filenames']]
    
    return images, fine_labels, coarse_labels, filenames


def load_cifar10n(path: Path) -> Dict[str, np.ndarray]:
    """Load CIFAR-10-N human annotation data."""
    data = np.load(path, allow_pickle=True)
    return data.item()


def load_cifar100n(path: Path) -> Dict[str, np.ndarray]:
    """Load CIFAR-100-N human annotation data."""
    data = np.load(path, allow_pickle=True)
    return data.item()


# Load datasets
print("Loading CIFAR-10...")
cifar10_images, cifar10_labels, cifar10_filenames = load_cifar10(CIFAR10_DATA_PATH)
print(f"CIFAR-10: {cifar10_images.shape[0]} samples")

print("\nLoading CIFAR-100...")
cifar100_images, cifar100_fine_labels, cifar100_coarse_labels, cifar100_filenames = load_cifar100(CIFAR100_DATA_PATH)
print(f"CIFAR-100: {cifar100_images.shape[0]} samples")

print("\nLoading CIFAR-10-N...")
cifar10n_data = load_cifar10n(CIFAR10N_PATH)
print(f"CIFAR-10-N keys: {list(cifar10n_data.keys())}")

print("\nLoading CIFAR-100-N...")
cifar100n_data = load_cifar100n(CIFAR100N_PATH)
print(f"CIFAR-100-N keys: {list(cifar100n_data.keys())}")

Loading CIFAR-10...
CIFAR-10: 50000 samples

Loading CIFAR-100...
CIFAR-100: 50000 samples

Loading CIFAR-10-N...
CIFAR-10-N keys: ['clean_label', 'aggre_label', 'worse_label', 'random_label1', 'random_label2', 'random_label3']

Loading CIFAR-100-N...
CIFAR-100-N keys: ['clean_label', 'noise_label']


## Anomaly Type Injection Functions

In [5]:
from enum import Enum

class AnomalyType(Enum):
    CLEAN = "clean"
    NEAR_MISS = "near_miss"
    GROSS = "gross"
    OOD = "ood"
    CLEAN_HARD = "clean_hard"
    RANDOM_FLIP = "random_flip"


def inject_near_miss_noise(label: int, near_miss_pairs: Dict[int, List[int]], num_classes: int) -> int:
    """Inject near-miss label noise by flipping to a semantically similar class."""
    similar_classes = near_miss_pairs.get(label, [])
    if similar_classes:
        return random.choice(similar_classes)
    # Fallback: random flip if no similar classes defined
    other_classes = [i for i in range(num_classes) if i != label]
    return random.choice(other_classes)


def inject_gross_noise(label: int, gross_pairs: Dict[int, List[int]], num_classes: int) -> int:
    """Inject gross label noise by flipping to a semantically distant class."""
    distant_classes = gross_pairs.get(label, [])
    if distant_classes:
        return random.choice(distant_classes)
    # Fallback: random flip to a very different class
    other_classes = [i for i in range(num_classes) if i != label]
    return random.choice(other_classes)


def inject_random_flip(label: int, num_classes: int) -> int:
    """Randomly flip label to any other class with equal probability."""
    other_classes = [i for i in range(num_classes) if i != label]
    return random.choice(other_classes)


def generate_ood_image(original_image: np.ndarray, img_shape: Tuple[int, int] = (32, 32, 3)) -> np.ndarray:
    """Generate an OOD image (random noise or from a different distribution)."""
    # Option 1: Random noise image
    # return np.random.randint(0, 256, size=(3072,), dtype=np.uint8)
    
    # Option 2: Blurred/destroyed version of original (simpler for now)
    # For a proper OOD, you might want to use SVHN or another dataset
    return np.random.randint(0, 256, size=(3072,), dtype=np.uint8)


def identify_clean_hard_samples(images: np.ndarray, labels: np.ndarray, 
                                 clean_hard_ratio: float = 0.1) -> np.ndarray:
    """
    Identify clean-but-hard samples.
    
    In practice, this would require a pre-trained model to find samples that are
    correctly labeled but difficult to classify (high loss, low confidence).
    
    For now, we use a heuristic: samples that are at class boundaries
    (using simple distance to class centroids).
    """
    num_samples = len(labels)
    num_hard = int(num_samples * clean_hard_ratio)
    
    # Simple heuristic: random selection (would be replaced with model-based selection)
    # In the full pipeline, this would use a pre-trained model's predictions
    hard_indices = np.random.choice(num_samples, size=num_hard, replace=False)
    
    is_hard = np.zeros(num_samples, dtype=bool)
    is_hard[hard_indices] = True
    
    return is_hard

## Main Dataset Generation Function

In [6]:
def generate_synthetic_noisy_dataset(
    images: np.ndarray,
    labels: np.ndarray,
    filenames: List[str],
    cifarN_data: Optional[Dict[str, np.ndarray]],
    near_miss_pairs: Dict[int, List[int]],
    gross_pairs: Dict[int, List[int]],
    num_classes: int,
    dataset_name: str,
    anomaly_ratios: Dict[str, float] = None,
    use_human_labels: bool = True
) -> pd.DataFrame:
    """
    Generate a synthetically noisy dataset with various anomaly types.
    
    Args:
        images: Image data array
        labels: Original clean labels
        filenames: List of filenames
        cifarN_data: CIFAR-N human annotation data (optional)
        near_miss_pairs: Dictionary mapping labels to semantically similar labels
        gross_pairs: Dictionary mapping labels to semantically distant labels
        num_classes: Number of classes in the dataset
        dataset_name: Name of the dataset (cifar10 or cifar100)
        anomaly_ratios: Dictionary specifying ratio of each anomaly type
        use_human_labels: Whether to include human label column from CIFAR-N
    
    Returns:
        DataFrame with metadata and label mappings
    """
    if anomaly_ratios is None:
        # Default anomaly ratios
        anomaly_ratios = {
            'clean': 0.40,
            'near_miss': 0.15,
            'gross': 0.10,
            'ood': 0.05,
            'clean_hard': 0.15,
            'random_flip': 0.15,
        }
    
    num_samples = len(labels)
    
    # Verify ratios sum to 1
    total_ratio = sum(anomaly_ratios.values())
    if abs(total_ratio - 1.0) > 0.001:
        raise ValueError(f"Anomaly ratios must sum to 1.0, got {total_ratio}")
    
    # Calculate number of samples for each anomaly type
    anomaly_counts = {k: int(v * num_samples) for k, v in anomaly_ratios.items()}
    
    # Adjust for rounding errors
    total_assigned = sum(anomaly_counts.values())
    if total_assigned < num_samples:
        anomaly_counts['clean'] += num_samples - total_assigned
    
    print(f"\n{dataset_name.upper()} Anomaly Distribution:")
    for anomaly_type, count in anomaly_counts.items():
        print(f"  {anomaly_type}: {count} samples ({count/num_samples*100:.1f}%)")
    
    # Randomly assign anomaly types to samples
    indices = np.arange(num_samples)
    np.random.shuffle(indices)
    
    # Assign anomaly types
    anomaly_assignments = {}
    start_idx = 0
    for anomaly_type, count in anomaly_counts.items():
        anomaly_assignments[anomaly_type] = indices[start_idx:start_idx + count]
        start_idx += count
    
    # Create metadata arrays
    sample_indices = []
    original_labels = []
    synthetic_labels = []
    anomaly_types = []
    human_labels = []
    human_label_available = []
    filenames_list = []
    
    # Process each sample
    for anomaly_type, type_indices in anomaly_assignments.items():
        for idx in type_indices:
            sample_indices.append(idx)
            original_labels.append(labels[idx])
            filenames_list.append(filenames[idx])
            
            # Get human label if available
            if use_human_labels and cifarN_data is not None:
                if dataset_name == 'cifar10':
                    # Use 'aggre_label' as the aggregated human label
                    h_label = cifarN_data.get('aggre_label', [None] * num_samples)[idx]
                    human_labels.append(h_label)
                    human_label_available.append(True)
                elif dataset_name == 'cifar100':
                    # Use 'noise_label' as the human label
                    h_label = cifarN_data.get('noise_label', [None] * num_samples)[idx]
                    human_labels.append(h_label)
                    human_label_available.append(True)
            else:
                human_labels.append(None)
                human_label_available.append(False)
            
            # Apply anomaly type
            if anomaly_type == 'clean':
                synthetic_labels.append(labels[idx])
                anomaly_types.append(AnomalyType.CLEAN.value)
            
            elif anomaly_type == 'near_miss':
                new_label = inject_near_miss_noise(labels[idx], near_miss_pairs, num_classes)
                synthetic_labels.append(new_label)
                anomaly_types.append(AnomalyType.NEAR_MISS.value)
            
            elif anomaly_type == 'gross':
                new_label = inject_gross_noise(labels[idx], gross_pairs, num_classes)
                synthetic_labels.append(new_label)
                anomaly_types.append(AnomalyType.GROSS.value)
            
            elif anomaly_type == 'ood':
                # OOD: random label (image would be replaced in actual dataset)
                ood_label = random.randint(0, num_classes - 1)
                synthetic_labels.append(ood_label)
                anomaly_types.append(AnomalyType.OOD.value)
            
            elif anomaly_type == 'clean_hard':
                # Clean but hard: correct label, but marked as hard
                synthetic_labels.append(labels[idx])
                anomaly_types.append(AnomalyType.CLEAN_HARD.value)
            
            elif anomaly_type == 'random_flip':
                new_label = inject_random_flip(labels[idx], num_classes)
                synthetic_labels.append(new_label)
                anomaly_types.append(AnomalyType.RANDOM_FLIP.value)
    
    # Create DataFrame
    df = pd.DataFrame({
        'sample_index': sample_indices,
        'filename': filenames_list,
        'original_label': original_labels,
        'synthetic_label': synthetic_labels,
        'anomaly_type': anomaly_types,
        'human_label': human_labels,
        'human_label_available': human_label_available,
    })
    
    # Sort by sample index for consistency
    df = df.sort_values('sample_index').reset_index(drop=True)
    
    # Add class name columns
    class_names = CIFAR10_CLASSES if dataset_name == 'cifar10' else CIFAR100_CLASSES
    df['original_class_name'] = df['original_label'].apply(lambda x: class_names[x])
    df['synthetic_class_name'] = df['synthetic_label'].apply(lambda x: class_names[x])
    
    # Add human class name if available
    if use_human_labels and cifarN_data is not None:
        df['human_class_name'] = df['human_label'].apply(
            lambda x: class_names[x] if x is not None else None
        )
    
    return df

## Generate Synthetic Noisy CIFAR-10

In [7]:
# Generate CIFAR-10 synthetic noisy dataset
print("=" * 60)
print("GENERATING SYNTHETIC NOISY CIFAR-10 DATASET")
print("=" * 60)

cifar10_df = generate_synthetic_noisy_dataset(
    images=cifar10_images,
    labels=cifar10_labels,
    filenames=cifar10_filenames,
    cifarN_data=cifar10n_data,
    near_miss_pairs=CIFAR10_NEAR_MISS_PAIRS,
    gross_pairs=CIFAR10_GROSS_NOISE_PAIRS,
    num_classes=10,
    dataset_name='cifar10',
    use_human_labels=True
)

print(f"\nGenerated CIFAR-10 dataset with {len(cifar10_df)} samples")
print("\nSample rows:")
print(cifar10_df.head(10))

print("\nAnomaly type distribution:")
print(cifar10_df['anomaly_type'].value_counts())

GENERATING SYNTHETIC NOISY CIFAR-10 DATASET

CIFAR10 Anomaly Distribution:
  clean: 20000 samples (40.0%)
  near_miss: 7500 samples (15.0%)
  gross: 5000 samples (10.0%)
  ood: 2500 samples (5.0%)
  clean_hard: 7500 samples (15.0%)
  random_flip: 7500 samples (15.0%)

Generated CIFAR-10 dataset with 50000 samples

Sample rows:
   sample_index                                  filename  original_label  \
0             0  leptodactylus_pentadactylus_s_000004.png               6   
1             1                       camion_s_000148.png               9   
2             2                 tipper_truck_s_001250.png               9   
3             3                 american_elk_s_001521.png               4   
4             4                station_wagon_s_000293.png               1   
5             5                        coupe_s_001735.png               1   
6             6                    cassowary_s_001300.png               2   
7             7                     cow_pony_s_001168.p

## Generate Synthetic Noisy CIFAR-100

In [8]:
# Generate CIFAR-100 synthetic noisy dataset
print("=" * 60)
print("GENERATING SYNTHETIC NOISY CIFAR-100 DATASET")
print("=" * 60)

cifar100_df = generate_synthetic_noisy_dataset(
    images=cifar100_images,
    labels=cifar100_fine_labels,
    filenames=cifar100_filenames,
    cifarN_data=cifar100n_data,
    near_miss_pairs=CIFAR100_NEAR_MISS_PAIRS,
    gross_pairs={},  # Will use random distant classes for CIFAR-100
    num_classes=100,
    dataset_name='cifar100',
    use_human_labels=True
)

print(f"\nGenerated CIFAR-100 dataset with {len(cifar100_df)} samples")
print("\nSample rows:")
print(cifar100_df.head(10))

print("\nAnomaly type distribution:")
print(cifar100_df['anomaly_type'].value_counts())

GENERATING SYNTHETIC NOISY CIFAR-100 DATASET

CIFAR100 Anomaly Distribution:
  clean: 20000 samples (40.0%)
  near_miss: 7500 samples (15.0%)
  gross: 5000 samples (10.0%)
  ood: 2500 samples (5.0%)
  clean_hard: 7500 samples (15.0%)
  random_flip: 7500 samples (15.0%)

Generated CIFAR-100 dataset with 50000 samples

Sample rows:
   sample_index                     filename  original_label  synthetic_label  \
0             0      bos_taurus_s_000507.png              19               22   
1             1     stegosaurus_s_000125.png              29               64   
2             2        mcintosh_s_000643.png               0               21   
3             3       altar_boy_s_001435.png              11               11   
4             4         cichlid_s_000031.png               1                1   
5             5           phone_s_002161.png              86               71   
6             6       car_train_s_000043.png              90               90   
7             7     

## Cross-Reference Analysis with CIFAR-N Human Labels

In [9]:
def analyze_human_label_agreement(df: pd.DataFrame, dataset_name: str):
    """Analyze agreement between synthetic labels and human labels."""
    
    # Filter samples where human labels are available
    df_human = df[df['human_label_available'] == True].copy()
    
    # Check agreement between original and human labels
    df_human['original_matches_human'] = df_human['original_label'] == df_human['human_label']
    df_human['synthetic_matches_human'] = df_human['synthetic_label'] == df_human['human_label']
    
    print(f"\n{'='*60}")
    print(f"HUMAN LABEL AGREEMENT ANALYSIS ({dataset_name.upper()})")
    print(f"{'='*60}")
    
    print(f"\nSamples with human labels: {len(df_human)} / {len(df)} ({len(df_human)/len(df)*100:.1f}%)")
    
    original_agreement = df_human['original_matches_human'].mean() * 100
    synthetic_agreement = df_human['synthetic_matches_human'].mean() * 100
    
    print(f"\nOriginal label matches human label: {original_agreement:.2f}%")
    print(f"Synthetic label matches human label: {synthetic_agreement:.2f}%")
    
    # Agreement by anomaly type
    print(f"\nSynthetic label agreement by anomaly type:")
    agreement_by_type = df_human.groupby('anomaly_type')['synthetic_matches_human'].mean() * 100
    print(agreement_by_type)
    
    return df_human


# Analyze CIFAR-10
cifar10_human_analysis = analyze_human_label_agreement(cifar10_df, 'cifar10')

# Analyze CIFAR-100
cifar100_human_analysis = analyze_human_label_agreement(cifar100_df, 'cifar100')


HUMAN LABEL AGREEMENT ANALYSIS (CIFAR10)

Samples with human labels: 50000 / 50000 (100.0%)

Original label matches human label: 9.79%
Synthetic label matches human label: 9.86%

Synthetic label agreement by anomaly type:
anomaly_type
clean           9.82
clean_hard     10.00
gross           9.80
near_miss       9.72
ood             9.84
random_flip    10.04
Name: synthetic_matches_human, dtype: float64

HUMAN LABEL AGREEMENT ANALYSIS (CIFAR100)

Samples with human labels: 50000 / 50000 (100.0%)

Original label matches human label: 0.98%
Synthetic label matches human label: 0.93%

Synthetic label agreement by anomaly type:
anomaly_type
clean          0.940000
clean_hard     0.906667
gross          1.000000
near_miss      1.066667
ood            0.680000
random_flip    0.826667
Name: synthetic_matches_human, dtype: float64


## Save CSV Files

In [12]:
# Save CIFAR-10 metadata
cifar10_csv_path = OUTPUT_DIR / "cifar10_synthetic_noisy_metadata.csv"
cifar10_df.to_csv(cifar10_csv_path, index=False)
print(f"\nSaved CIFAR-10 metadata to: {cifar10_csv_path}")

# Save CIFAR-100 metadata
cifar100_csv_path = OUTPUT_DIR / "cifar100_synthetic_noisy_metadata.csv"
cifar100_df.to_csv(cifar100_csv_path, index=False)
print(f"Saved CIFAR-100 metadata to: {cifar100_csv_path}")


# - sample_index: Index of the sample in the original CIFAR dataset
# - filename: Original image filename
# - original_label: Clean ground truth label (class index)
# - original_class_name: Clean ground truth label (class name)
# - synthetic_label: Potentially noisy label after anomaly injection (class index)
# - synthetic_class_name: Potentially noisy label after anomaly injection (class name)
# - anomaly_type: Type of anomaly injected (clean, near_miss, gross, ood, clean_hard, random_flip)
# - human_label: Human-annotated label from CIFAR-N dataset (class index)
# - human_class_name: Human-annotated label from CIFAR-N dataset (class name)
# - human_label_available: Whether human label is available for this sample

# ANOMALY TYPES:
# - clean: Correct label, clean image
# - near_miss: Label flipped to semantically similar class
# - gross: Label flipped to semantically distant class
# - ood: Out-of-distribution image with random label
# - clean_hard: Correct label, but image is difficult to classify
# - random_flip: Label randomly flipped to any other class



Saved CIFAR-10 metadata to: /projectnb/ivc-ml/appledora/CS506/cs506-anomaly-det/data/cifar10_synthetic_noisy_metadata.csv
Saved CIFAR-100 metadata to: /projectnb/ivc-ml/appledora/CS506/cs506-anomaly-det/data/cifar100_synthetic_noisy_metadata.csv


## Verification and Summary Statistics

In [13]:
def print_dataset_summary(df: pd.DataFrame, dataset_name: str):
    """Print summary statistics for the generated dataset."""
    print(f"\n{'='*60}")
    print(f"DATASET SUMMARY: {dataset_name.upper()}")
    print(f"{'='*60}")
    
    print(f"\nTotal samples: {len(df)}")
    print(f"Number of classes: {df['original_label'].nunique()}")
    
    print(f"\nAnomaly Type Distribution:")
    anomaly_dist = df['anomaly_type'].value_counts()
    for anomaly_type, count in anomaly_dist.items():
        print(f"  {anomaly_type}: {count} ({count/len(df)*100:.1f}%)")
    
    print(f"\nOriginal Label Distribution:")
    print(df['original_class_name'].value_counts().head(5))
    
    print(f"\nSynthetic Label Distribution:")
    print(df['synthetic_class_name'].value_counts().head(5))
    
    if 'human_label' in df.columns:
        human_available = df['human_label_available'].sum()
        print(f"\nHuman labels available: {human_available} ({human_available/len(df)*100:.1f}%)")


print_dataset_summary(cifar10_df, 'cifar10')
print_dataset_summary(cifar100_df, 'cifar100')


DATASET SUMMARY: CIFAR10

Total samples: 50000
Number of classes: 10

Anomaly Type Distribution:
  clean: 20000 (40.0%)
  clean_hard: 7500 (15.0%)
  near_miss: 7500 (15.0%)
  random_flip: 7500 (15.0%)
  gross: 5000 (10.0%)
  ood: 2500 (5.0%)

Original Label Distribution:
original_class_name
frog          5000
truck         5000
deer          5000
automobile    5000
bird          5000
Name: count, dtype: int64

Synthetic Label Distribution:
synthetic_class_name
dog         5628
airplane    5585
deer        5552
bird        5165
truck       5110
Name: count, dtype: int64

Human labels available: 50000 (100.0%)

DATASET SUMMARY: CIFAR100

Total samples: 50000
Number of classes: 100

Anomaly Type Distribution:
  clean: 20000 (40.0%)
  random_flip: 7500 (15.0%)
  clean_hard: 7500 (15.0%)
  near_miss: 7500 (15.0%)
  gross: 5000 (10.0%)
  ood: 2500 (5.0%)

Original Label Distribution:
original_class_name
cattle      500
rocket      500
tiger       500
flatfish    500
fox         500
Name: co

## Example: Inspect Specific Samples

In [14]:
# Show examples of each anomaly type for CIFAR-10
print("\n" + "=" * 60)
print("EXAMPLE SAMPLES BY ANOMALY TYPE (CIFAR-10)")
print("=" * 60)

for anomaly_type in cifar10_df['anomaly_type'].unique():
    print(f"\n--- {anomaly_type.upper()} ---")
    examples = cifar10_df[cifar10_df['anomaly_type'] == anomaly_type].head(3)
    for _, row in examples.iterrows():
        human_info = f"Human: {row['human_class_name']}" if row['human_label_available'] else "Human: N/A"
        print(f"  Index {row['sample_index']}: {row['original_class_name']} -> {row['synthetic_class_name']} | {human_info}")


EXAMPLE SAMPLES BY ANOMALY TYPE (CIFAR-10)

--- CLEAN ---
  Index 0: frog -> frog | Human: horse
  Index 1: truck -> truck | Human: ship
  Index 4: automobile -> automobile | Human: frog

--- CLEAN_HARD ---
  Index 2: truck -> truck | Human: deer
  Index 5: automobile -> automobile | Human: dog
  Index 10: deer -> deer | Human: truck

--- NEAR_MISS ---
  Index 3: deer -> bird | Human: deer
  Index 8: ship -> truck | Human: frog
  Index 14: truck -> ship | Human: ship

--- RANDOM_FLIP ---
  Index 9: cat -> frog | Human: frog
  Index 16: truck -> cat | Human: truck
  Index 55: bird -> frog | Human: frog

--- GROSS ---
  Index 18: bird -> truck | Human: deer
  Index 24: bird -> ship | Human: bird
  Index 25: frog -> automobile | Human: ship

--- OOD ---
  Index 19: frog -> ship | Human: truck
  Index 42: bird -> horse | Human: deer
  Index 50: truck -> deer | Human: cat


## Save Additional Metadata (Optional)

In [15]:
# Save anomaly type mappings for reference
import json

metadata = {
    'cifar10_classes': CIFAR10_CLASSES,
    'cifar100_classes': CIFAR100_CLASSES,
    'cifar10_near_miss_pairs': {str(k): v for k, v in CIFAR10_NEAR_MISS_PAIRS.items()},
    'cifar10_gross_pairs': {str(k): v for k, v in CIFAR10_GROSS_NOISE_PAIRS.items()},
    'cifar100_coarse_labels': CIFAR100_COARSE_LABELS,
    'random_seed': RANDOM_SEED,
    'default_anomaly_ratios': {
        'clean': 0.40,
        'near_miss': 0.15,
        'gross': 0.10,
        'ood': 0.05,
        'clean_hard': 0.15,
        'random_flip': 0.15,
    }
}

metadata_path = OUTPUT_DIR / "dataset_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Saved dataset metadata to: {metadata_path}")

Saved dataset metadata to: /projectnb/ivc-ml/appledora/CS506/cs506-anomaly-det/data/dataset_metadata.json


## Summary

This notebook has successfully generated synthetically noisy CIFAR-10 and CIFAR-100 datasets with the following features:

1. **Anomaly Types Injected**:
   - Clean (40%): Correct labels
   - Near-miss (15%): Semantically similar label flips
   - Gross (10%): Semantically distant label flips
   - OOD (5%): Out-of-distribution samples
   - Clean-hard (15%): Difficult but correctly labeled samples
   - Random flip (15%): Randomly flipped labels

2. **Human Label Cross-Reference**:
   - All samples are cross-referenced with CIFAR-10/100-N human annotations
   - Human labels are included as an additional column when available
   - Agreement analysis between synthetic and human labels

3. **Output Files**:
   - `cifar10_synthetic_noisy_metadata.csv`: Complete metadata for CIFAR-10
   - `cifar100_synthetic_noisy_metadata.csv`: Complete metadata for CIFAR-100
   - `dataset_metadata.json`: Class names and semantic mappings

### Next Steps

1. Train clean ViT model on clean samples only
2. Train noisy ViT model on full dataset with dynamics logging
3. Extract geometry, uncertainty, and training dynamics features
4. Test hypotheses H1, H2, and H3